In [1]:
%%capture
!pip install "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install bnlp_toolkit nltk rouge-score datasets
!pip install peft accelerate bitsandbytes

In [ ]:
!pip install unsloth_zoo

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True" 

import unsloth
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from transformers import TrainingArguments
from trl import SFTTrainer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-02-24 19:59:36.543958: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771963176.762784      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771963176.829363      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771963177.437469      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771963177.437522      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771963177.437526      55 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
# !pip show unsloth

In [5]:
print(torch.cuda.is_available())
print(torch.version.cuda)

True
12.6


# Downloading Dataset

In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("raseluddin/bengali-empathetic-conversations-corpus")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/raseluddin/bengali-empathetic-conversations-corpus


## Loading the Dataset

In [7]:
import pandas as pd
import os

csv_path = os.path.join(path, "BengaliEmpatheticConversationsCorpus .csv")
empathic_df = pd.read_csv(csv_path)

empathic_df.head()

,Topics,Question-Title,Questions,Answers
0,পারিবারিক দ্বন্দ্ব,মা ও স্ত্রীর মধ্যে মতানৈক্য বৃদ্ধি,আমার স্ত্রী এবং মায়ের মধ্যে টানটান মতবিরোধ চ...,"আপনি যা বর্ণনা করছেন তাকে মনোবিজ্ঞানীরা ""ত্রি..."
1,"পদার্থের অপব্যবহার, আসক্তি",আমি ধূমপানে আসক্ত। আমি কিভাবে থামাতে পারি?,"আমি বাচ্চা নেওয়ার পরিকল্পনা করছি, তাই আমাকে ...",হাই। আপনার শিশুর (এবং নিজের) জন্য যা স্বাস্থ্...
2,পারিবারিক দ্বন্দ্ব,আমার পরিবারের কাছ থেকে গোপন রাখা,"আমার মনের মধ্যে গোপন আছে, এবং আমি জানি না তাদ...",মনে হচ্ছে গোপন রাখা এখন আপনার জন্য একটি সমস্য...
3,"আচরণগত পরিবর্তন, সামাজিক সম্পর্ক",অধিকারী হওয়ার অন্তর্নিহিত কারণ,আমি আমার সম্পর্কের ক্ষেত্রে অত্যন্ত অধিকারসূচক...,হ্যালো। এটা দুর্দান্ত যে আপনি উপলব্ধি করতে সক...
4,দুশ্চিন্তা,আমি কি ওষুধ ছাড়া উদ্বেগ নিয়ন্ত্রণ করতে পারি?,কয়েক বছর আগে আমার মাথায় আঘাত লেগেছিল এবং আমা...,আপনি বলেননি কি বা কত ওষুধ আপনি চেষ্টা করেছেন।...


# Constants

In [8]:
DATA_PATH = csv_path
DB_PATH = "experiments.db"
MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'

# DatasetProcessor => Preprocessing

In [9]:
from datasets import Dataset
from transformers import AutoTokenizer


class DatasetProcessor:
    """
    Handles data loading, chat template formatting, and tokenization 
    for Bengali Empathetic Conversations dataset.
    """
    def __init__(self, model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit", filter_out=768):
        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.filter_out = filter_out
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
            
    def load_data(self, file_path: str) -> Dataset:
        """Loads the CSV file into a Hugging Face Dataset object."""
        try:
            df = pd.read_csv(file_path)

            # DROP MISSINGS
            initial_len = len(df)
            df = df.dropna(subset=['Topics', 'Question-Title', 'Questions', 'Answers'])
            
            # if "Questions" not in df.columns or "Answers" not in df.columns:
            #     raise ValueError("Dataset is missing required 'Questions' or 'Answers' columns.")
            return Dataset.from_pandas(df)
            
        except Exception as e:
            print(f"Error loading data: {e}")
            raise
            

    def apply_llama_template(self, example):
        """
        Maps the conversational data into LLaMA 3.1's format, 
        utilizing Topics for the system prompt and Question-Title for context.
        """

        topic = example.get("Topics", "General Conversation")
        system_prompt = f"You are a highly empathetic and helpful AI assistant conversing in Bengali. The primary topic of this conversation is: {topic}."
        
        title = example.get("Question-Title", "")
        question = example.get("Questions", "")

        user_content = f"Context: {title}\nQuestion: {question}" if title else question

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": example.get("Answers", "")}
        ]
        
        # apply_the_chat_template (adds <|start_header_id|>, <|eot_id|>, etc.)
        formatted_text = self.tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        return {"formatted_text": formatted_text}
        

    def tokenize_text(self, example):
        return self.tokenizer(
            example["formatted_text"],
            truncation=False,  # SEQUENCE LENGTH WILL NOT BE REDUCED
            padding=False,     # WILL USE DYNAMIC MEMORY PADDING TO SAVE MEMORY
            add_special_tokens=False # apply_chat_template ALREADY HANDLES SPECIAL TOKENS
        )
        

    def filter_long_sequences(self, example):
        tokens = self.tokenizer(example["formatted_text"], truncation=False, add_special_tokens=False)
        return len(tokens["input_ids"]) <= self.filter_out

    def process(self, file_path: str):
        dataset = self.load_data(file_path)
        dataset = dataset.map(self.apply_llama_template, num_proc=2)
        
        # FILTER OUT TOKENS WITH CUSTOM ARGUMENT => self.filter_out
        print(f"Filtering out rows exceeding {self.filter_out} tokens to prevent GPU OOM...")
        dataset = dataset.filter(self.filter_long_sequences, num_proc=2)
        
        return dataset.map(self.tokenize_text, batched=True, num_proc=2)

# Example Usage:
# processor = DatasetProcessor()
# train_dataset = processor.process("bengali_empathetic_conversations.csv")

# Count Original Max Sequence Length

In [24]:
import numpy as np
from tqdm import tqdm

print("\n--- Running Sequence Length Analysis ===")
lengths = []

processor = DatasetProcessor(MODEL_NAME, 768)
full_dataset = processor.process(DATA_PATH) 

for example in tqdm(full_dataset, desc="Calculating Token Lengths"):
    tokens = processor.tokenizer(
        example["formatted_text"],
        truncation=False,
        add_special_tokens=False
    )
    
    lengths.append(len(tokens["input_ids"]))

lengths = np.array(lengths)

print(f"Max length: {lengths.max()}")
print(f"Mean length: {lengths.mean():.2f}")
print(f"95th percentile: {np.percentile(lengths, 95):.0f}")
print(f"99th percentile: {np.percentile(lengths, 99):.0f}")
print("================================xxxx===============================\n")


--- Running Sequence Length Analysis ===


Map (num_proc=2):   0%|          | 0/37610 [00:00<?, ? examples/s]

Filtering out rows exceeding 768 tokens to prevent GPU OOM...


Filter (num_proc=2):   0%|          | 0/37610 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/36150 [00:00<?, ? examples/s]

Calculating Token Lengths: 100%|██████████| 36150/36150 [00:32<00:00, 1100.79it/s]


Max length: 768
Mean length: 257.40
95th percentile: 412
99th percentile: 552
================================xxxx===============================



# LLAMAFineTuner => Fine Tuning LLaMA 3.1 Instruct 8b

## Utility Classes for LLAMAFineTuner

In [12]:
from abc import ABC, abstractmethod
from trl import SFTTrainer


class FineTuningStrategy(ABC):
    """Abstract base class for fine-tuning strategies.
    created this to pass different kind of finetuning strategies if need be
    ensuring modularity."""
    
    @abstractmethod
    def setup_model(self, model_name: str, max_seq_length: int):
        """Loads the model and applies LoRA/quantization."""
        pass

    @abstractmethod
    def setup_training_args(self, output_dir: str) -> TrainingArguments:
        """Configures the training arguments for memory optimization."""
        pass


class UnslothStrategy(FineTuningStrategy):
    def setup_model(self, model_name: str, max_seq_length: int):
        print("Loading model via Unsloth with 4-bit quantization...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=max_seq_length,
            dtype=None,
            load_in_4bit=True,
            device_map={"": 0},
        )
        
        print("Applying Memory-Optimized LoRA...")
        model = FastLanguageModel.get_peft_model(
            model,
            r=8, 
            target_modules=["q_proj", "v_proj"], # TARGETING JUST Q & V TO SAVE VRAM
            lora_alpha=16,
            lora_dropout=0, 
            bias="none",
            use_gradient_checkpointing="unsloth", 
            random_state=3407,
            use_rslora=False,
            loftq_config=None,
        )
        return model, tokenizer

    def setup_training_args(self, output_dir: str) -> TrainingArguments:
        return TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=1, 
            gradient_accumulation_steps=8,
            warmup_steps=5,
            max_steps=50, 
            learning_rate=2e-4,
            fp16=True,
            bf16=False,
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=3407,
            group_by_length=True,
            report_to="none"
        )

## LLAMAFineTuner

In [13]:
class LLAMAFineTuner:
    def __init__(self, strategy: FineTuningStrategy, max_seq_length: int = 6784):
        self.strategy = strategy
        self.max_seq_length = max_seq_length
        self.model = None
        self.tokenizer = None
        
    def train(self, dataset, model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit", output_dir="lora_model"):
        """Runs the SFT (Supervised Fine-Tuning) pipeline."""
        
        # STEUP MODEL VIA STRATEGY
        self.model, self.tokenizer = self.strategy.setup_model(model_name, self.max_seq_length)
        
        # SETUP ARGUMENTS VIA STRATEGY
        training_args = self.strategy.setup_training_args(output_dir)
        
        print("Initializing SFTTrainer...")
        trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            train_dataset=dataset,
            dataset_text_field="formatted_text", # From our DatasetProcessor
            max_seq_length=self.max_seq_length,
            dataset_num_proc=2,
            packing=False, # False for variable length conversations
            args=training_args,
        )
        
        # Execute Training
        print("Starting training loop...")
        trainer_stats = trainer.train()
        print(f"Training complete. Memory used: {trainer_stats.metrics['train_runtime']} seconds.")
        
        # 5. Save Model
        # self.model.save_pretrained_merged(output_dir, self.tokenizer, save_method="lora") # SAVES THE BASE MODEL (pretrained)
        
        # (Saves only the adapters)
        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)
        print(f"LoRA adapters saved to {output_dir}")
        
        return trainer_stats, training_args.output_dir

        
# Example Usage:
# my_strategy = UnslothStrategy()
# tuner = LLAMAFineTuner(strategy=my_strategy, max_seq_length=6784)
# stats = tuner.train(train_dataset)

# Evaluator

In [14]:
import sqlite3
import json
import math
from datetime import datetime
from bnlp import BasicTokenizer
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
from tqdm import tqdm


class Evaluator:
    """
    Handles model evaluation (Perplexity, BLEU, ROUGE) and 
    logs results to a local SQLite database.
    """
    def __init__(self, model, tokenizer, db_path=DB_PATH):
        self.model = model
        self.tokenizer = tokenizer
        self.bnlp_tokenizer = BasicTokenizer()
        self.rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
        self.db_path = db_path
        self._setup_database()

    def _setup_database(self):
        """Creates the required tables if they don't exist."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Table 1: LLAMAExperiments
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS LLAMAExperiments (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                model_name TEXT,
                lora_config TEXT,
                train_loss REAL,
                val_loss REAL,
                metrics TEXT,
                timestamp TEXT
            )
        ''')
        
        # Table 2: GeneratedResponses
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS GeneratedResponses (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                experiment_id INTEGER,
                input_text TEXT,
                response_text TEXT,
                timestamp TEXT,
                FOREIGN KEY(experiment_id) REFERENCES LLAMAExperiments(id)
            )
        ''')
        conn.commit()
        conn.close()

        
    def calculate_perplexity(self, test_dataset):
        """Calculates Perplexity over the test dataset with a progress bar."""
        print("Calculating Perplexity...")
        self.model.eval()
        nlls = []
        
        # ADDED TQDM PROGRESS BAR HERE
        for example in tqdm(test_dataset, desc="Evaluating Perplexity"):
            encodings = self.tokenizer(example["formatted_text"], return_tensors="pt").to("cuda")
            
            if encodings.input_ids.size(1) < 2:
                continue
                
            with torch.no_grad():
                outputs = self.model(encodings.input_ids, labels=encodings.input_ids)
                neg_log_likelihood = outputs.loss

            nlls.append(neg_log_likelihood)

        if not nlls:
            return float("inf")
            
        ppl = torch.exp(torch.stack(nlls).mean())
        return ppl.item()
    

    def evaluate_generation(self, test_dataset, experiment_id):
        bleu_scores = []
        rouge_scores = {'rouge1': 0, 'rouge2': 0, 'rougeL': 0}
        

        FastLanguageModel.for_inference(self.model) 
        
        print("Generating responses for BLEU/ROUGE calculation...")

        
        for example in tqdm(test_dataset, desc="Generating & Scoring"):
            prompt = example["formatted_text"].split("<|start_header_id|>assistant<|end_header_id|>\n\n")[0] + "<|start_header_id|>assistant<|end_header_id|>\n\n"
            inputs = self.tokenizer([prompt], return_tensors="pt").to("cuda")
            
            outputs = self.model.generate(**inputs, max_new_tokens=100, use_cache=True) # Reduced to 100 tokens for speed
            generated_text = self.tokenizer.batch_decode(outputs)[0].split("<|start_header_id|>assistant<|end_header_id|>\n\n")[-1].replace("<|eot_id|>", "").strip()
            
            reference_text = str(example.get("Answers", ""))
            
            # Log to DB
            self._log_response(experiment_id, example.get("Questions", ""), generated_text)

            # BNLP Tokenization for accurate Bengali metrics
            gen_tokens = self.bnlp_tokenizer(generated_text)
            ref_tokens = self.bnlp_tokenizer(reference_text)

            if not ref_tokens or not gen_tokens: continue

            bleu_scores.append(sentence_bleu([ref_tokens], gen_tokens, weights=(0.5, 0.5, 0, 0)))
            rouge_result = self.rouge.score(" ".join(ref_tokens), " ".join(gen_tokens))
            rouge_scores['rouge1'] += rouge_result['rouge1'].fmeasure
            rouge_scores['rougeL'] += rouge_result['rougeL'].fmeasure

        n = max(len(bleu_scores), 1)
        return {"BLEU": sum(bleu_scores)/n, "ROUGE-1": rouge_scores['rouge1']/n, "ROUGE-L": rouge_scores['rougeL']/n}

    
    def _log_response(self, exp_id, input_text, response_text):
        conn = sqlite3.connect(self.db_path); cursor = conn.cursor()
        cursor.execute('INSERT INTO GeneratedResponses (experiment_id, input_text, response_text, timestamp) VALUES (?, ?, ?, ?)', 
                       (exp_id, input_text, response_text, datetime.now().isoformat()))
        conn.commit(); conn.close()

    def log_experiment(self, model_name, lora_config, train_loss, val_loss, metrics):
        conn = sqlite3.connect(self.db_path); cursor = conn.cursor()
        cursor.execute('INSERT INTO LLAMAExperiments (model_name, lora_config, train_loss, val_loss, metrics, timestamp) VALUES (?, ?, ?, ?, ?, ?)', 
                       (model_name, json.dumps(lora_config), train_loss, val_loss, json.dumps(metrics), datetime.now().isoformat()))
        exp_id = cursor.lastrowid
        conn.commit(); conn.close()
        return exp_id
        

    def export_human_eval_csv(self, experiment_id, output_filename="human_evaluation_set.csv"):
        """Exports generations to a CSV for human reviewers."""
        conn = sqlite3.connect(self.db_path)
        query = f"SELECT input_text, response_text FROM GeneratedResponses WHERE experiment_id = {experiment_id}"
        df = pd.read_sql_query(query, conn)
        df["Empathetic_Score (1-5)"] = "" # Empty column for reviewers (HUMAN EVALUATION)
        df["Reviewer_Notes"] = ""
        df.to_csv(output_filename, index=False)
        conn.close()
        print(f"Exported human evaluation template to {output_filename}")

# Train & Execution

In [15]:
import gc


print("=== 1. Processing Dataset ===")
processor = DatasetProcessor(MODEL_NAME, 768)

full_dataset = processor.process(DATA_PATH)

filter_out = 768
print(f"Filtering sequences > {filter_out} tokens for T4 Hardware Limits...")
full_dataset = full_dataset.filter(lambda x: len(x["input_ids"]) <= filter_out, num_proc=2)

split_dataset = full_dataset.train_test_split(test_size=0.1, seed=42)
train_data = split_dataset['train']
test_data = split_dataset['test']

print("\n--- 2. Starting Fine-Tuning ---")
strategy = UnslothStrategy()
tuner = LLAMAFineTuner(strategy=strategy, max_seq_length=filter_out) 

# FORCE PURGE ALL DEAD VRAM BEFORE TRAINING STARTS
gc.collect()
torch.cuda.empty_cache()

trainer_stats, output_dir = tuner.train(train_data)

=== 1. Processing Dataset ===


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Map (num_proc=2):   0%|          | 0/37610 [00:00<?, ? examples/s]

...

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch Attention layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.2.1 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Initializing SFTTrainer...
Starting training loop...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 32,535 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 3,407,872 of 8,033,669,120 (0.04% trained)


Step,Training Loss
10,1.390800
20,1.007000
30,0.699800
40,0.636100
50,0.614200


Training complete. Memory used: 379.3042 seconds.
LoRA adapters saved to lora_model


# Evaluatiton and Exporting Deliverables

In [16]:
import shutil
from IPython.display import FileLink
import zipfile
from pathlib import Path


print("\n--- 3. Running Standalone Evaluation & Logging ---")

# re-initialize the evaluator with the model that is already in memory
evaluator = Evaluator(model=tuner.model, tokenizer=tuner.tokenizer, db_path=DB_PATH)

# Take small random samples to speed up evaluation
samples = 100
print(f"Selecting {samples} random test samples")
fast_test_data = test_data.shuffle(seed=42).select(range(samples))

# Calculate Perplexity
ppl = evaluator.calculate_perplexity(fast_test_data)
print(f"Test Perplexity: {ppl:.4f}")

# Pre-log experiment to get an ID (if you don't already have one)
lora_cfg = {"r": 8, "alpha": 16, "target_modules": ["q_proj", "v_proj"]}

train_loss = trainer_stats.metrics.get('train_loss', 0.0) if 'trainer_stats' in locals() else 0.0

initial_metrics = {"perplexity": ppl}
exp_id = evaluator.log_experiment("Unsloth-LLaMA3.1-8B-Bengali", lora_cfg, train_loss, 0.0, initial_metrics)

# GENEARTE AND CALCULATE BLEU/ROUGE
print("Calculating BLEU and ROUGE Scores...")
final_metrics = evaluator.evaluate_generation(fast_test_data, exp_id)

# COMBINE AND UPDATE
final_metrics["perplexity"] = ppl 
print(f"Final Automated Metrics: {final_metrics}")

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("UPDATE LLAMAExperiments SET metrics = ? WHERE id = ?", (json.dumps(final_metrics), exp_id))
conn.commit()
conn.close()


print("\n--- 4. Exporting Deliverables ---")


OUTPUT_DIR = "submission_artifacts"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Ensured output directory exists: ./{OUTPUT_DIR}/")

# ROUTE ALL DELIVERABLES INTO THE DIRECTORY
csv_path = os.path.join(OUTPUT_DIR, "bengali_empathy_human_eval.csv")
db_path = os.path.join(OUTPUT_DIR, "experiments.db")

# Export CSV directly into the safe folder
evaluator.export_human_eval_csv(exp_id, csv_path)

# Copy the SQLite database into the safe folder
shutil.copy(DB_PATH, db_path)

# Save the LoRA weights into the safe folder (if they are in memory)
print("Saving LoRA adapter weights...")
tuner.model.save_pretrained(OUTPUT_DIR)
tuner.tokenizer.save_pretrained(OUTPUT_DIR)

# CUSTOM TQDM ZIPPING FUNCTION
def zip_directory_with_progress(folder_path, output_zip_path):
    folder_path = Path(folder_path)
    
    # Gather all files in the directory
    files_to_zip = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            files_to_zip.append(Path(root) / file)
            
    # Zip them with a tqdm progress bar
    with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file in tqdm(files_to_zip, desc="Zipping Deliverables", unit="file"):
            arcname = file.relative_to(folder_path) # Keeps internal folder structure clean
            zipf.write(file, arcname)

# EXECUTE ZIP
zip_filename = "final_submission.zip"
zip_directory_with_progress(OUTPUT_DIR, zip_filename)

print("\nEVALUATION & PACKAGING COMPLETE! Click the link below to download everything at once:")
display(FileLink(zip_filename))


--- 3. Running Standalone Evaluation & Logging ---
Selecting 100 random test samples
Calculating Perplexity...


Evaluating Perplexity: 100%|██████████| 100/100 [00:46<00:00,  2.16it/s]


Test Perplexity: 2.0647
Calculating BLEU and ROUGE Scores...
Generating responses for BLEU/ROUGE calculation...


Generating & Scoring: 100%|██████████| 100/100 [06:09<00:00,  3.70s/it]


Final Automated Metrics: {'BLEU': 0.009883279461163153, 'ROUGE-1': 0.0, 'ROUGE-L': 0.0, 'perplexity': 2.0646941661834717}

--- 4. Exporting Deliverables ---
Ensured output directory exists: ./submission_artifacts/
Exported human evaluation template to submission_artifacts/bengali_empathy_human_eval.csv
Saving LoRA adapter weights...


Zipping Deliverables: 100%|██████████| 9/9 [00:01<00:00,  7.89file/s]


EVALUATION & PACKAGING COMPLETE! Click the link below to download everything at once:


/kaggle/working/final_submission.zip

# Inference (Models response on test prompts)

In [21]:
test_cases = [
    {
        "test_topic": "পারিবারিক দ্বন্দ্ব",
        "test_title": "মা ও স্ত্রীর মধ্যে মতানৈক্য বৃদ্ধি",
        "test_question": "আমার স্ত্রী এবং মায়ের মধ্যে টানটান মতবিরোধ চলছে। আমি কি করতে পারি?"
    },
    {
        "test_topic": "শিক্ষাজনিত চাপ",
        "test_title": "পরীক্ষার আগে মানসিক চাপ বৃদ্ধি",
        "test_question": "পরীক্ষার সময় খুব বেশি চাপ অনুভব করছি। কীভাবে নিজেকে শান্ত রাখবো?"
    },
    {
        'test_topic': "প্রেম ও সম্পর্ক",
        'test_title': "সম্পর্কে বিশ্বাসের সংকট",
        'test_question': "আমি আমার সঙ্গীর উপর আগের মতো ভরসা করতে পারছি না। এই পরিস্থিতিতে কী করা উচিত?"
    },
    {
        "test_topic": "বন্ধুত্বের সমস্যা",
        "test_title": "ঘনিষ্ঠ বন্ধুর সাথে দূরত্ব তৈরি হওয়া",
        "test_question": "আমার খুব কাছের বন্ধুটি হঠাৎ করে দূরে সরে গেছে। আমি কি তার সাথে কথা বলবো নাকি অপেক্ষা করবো?"
    }
]

# Enable faster inference once
FastLanguageModel.for_inference(tuner.model)

# Iterate through test cases
for idx, case in enumerate(test_cases, 1):

    test_topic = case["test_topic"]
    test_title = case["test_title"]
    test_question = case["test_question"]

    system_prompt = (
        f"You are a highly empathetic and helpful AI assistant conversing in Bengali. "
        f"The primary topic of this conversation is: {test_topic}."
    )

    user_content = f"Context: {test_title}\nQuestion: {test_question}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]

    prompt = tuner.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tuner.tokenizer([prompt], return_tensors="pt").to("cuda")

    print(f"\n==============================")
    print(f"Generating response for Test Case {idx}...")
    print(f"Topic: {test_topic}")
    print(f"Title: {test_title}")
    print(f"==============================\n")

    outputs = tuner.model.generate(
        **inputs,
        max_new_tokens=150,
        use_cache=True
    )

    generated_text = tuner.tokenizer.batch_decode(outputs)[0]
    final_answer = generated_text.split(
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )[-1].replace("<|eot_id|>", "").strip()

    print("--- AI's Empathetic Response ---")
    print(final_answer)
    print("\n")


Generating response for Test Case 1...
Topic: পারিবারিক দ্বন্দ্ব
Title: মা ও স্ত্রীর মধ্যে মতানৈক্য বৃদ্ধি

--- AI's Empathetic Response ---
এটা খুব ব্যাপক এবং ক্লান্তিকর সময় হচ্ছে না তা আমি বুঝতে পারি। আপনি এবং আপনার স্ত্রী আপনার মায়ের সাথে কথা বলেন।



Generating response for Test Case 2...
Topic: শিক্ষাজনিত চাপ
Title: পরীক্ষার আগে মানসিক চাপ বৃদ্ধি

--- AI's Empathetic Response ---
আমি জানি যে পরীক্ষা বেশ চাপের সাথে আসে। আমি আপনাকে কিছু সুপার টিপ দিতে পারি: প্রথমে, আপনি শোকড়ায় নয়, আপনার প্রশ্নগুলি স�



Generating response for Test Case 3...
Topic: প্রেম ও সম্পর্ক
Title: সম্পর্কে বিশ্বাসের সংকট

--- AI's Empathetic Response ---
আমি আশা করি আমি এটা ভালোভাবে বুঝতে পারব না। আমি এটা বলতে চাই না যে আমি আপনাকে এটা বলতে পারি না।



Generating response for Test Case 4...
Topic: বন্ধুত্বের সমস্যা
Title: ঘনিষ্ঠ বন্ধুর সাথে দূরত্ব তৈরি হওয়া

--- AI's Empathetic Response ---
এটা স্বাভাবিক যে আপনি তার সাথে কথা বলবেন।




# Showcasing DB and Human Eval. CSV

## DB

In [18]:
import sqlite3
import pandas as pd

print("\n==== Experiment Database Records ===")

db_path = "submission_artifacts/experiments.db"
conn = sqlite3.connect(db_path)

query = "SELECT id, model_name, lora_config, train_loss, val_loss, metrics, timestamp FROM LLAMAExperiments"
df_experiments = pd.read_sql_query(query, conn)
conn.close()

# Display the formatted table
display(df_experiments)


==== Experiment Database Records ===


,id,model_name,lora_config,train_loss,val_loss,metrics,timestamp
0,1,Unsloth-LLaMA3.1-8B-Bengali,"{""r"": 8, ""alpha"": 16, ""target_modules"": [""q_pr...",0.869574,0.0,"{""BLEU"": 0.009883279461163153, ""ROUGE-1"": 0.0,...",2026-02-24T20:09:42.024502


# Evaluation metrics table and analysis

In [19]:
import sqlite3
import pandas as pd
import json

conn = sqlite3.connect("submission_artifacts/experiments.db")
query = "SELECT train_loss, val_loss, metrics FROM LLAMAExperiments ORDER BY id DESC LIMIT 1"
df = pd.read_sql_query(query, conn)
conn.close()

latest_run = df.iloc[0]
metrics_dict = json.loads(latest_run['metrics'])

metrics_dict['Final Train Loss'] = round(latest_run['train_loss'], 4)
metrics_dict['Final Val Loss'] = round(latest_run['val_loss'], 4) if pd.notnull(latest_run['val_loss']) else "N/A"

results_df = pd.DataFrame(list(metrics_dict.items()), columns=['Metric', 'Score'])
print(results_df.to_markdown(index=False))

| Metric           |      Score |
|:-----------------|-----------:|
| BLEU             | 0.00988328 |
| ROUGE-1          | 0          |
| ROUGE-L          | 0          |
| perplexity       | 2.06469    |
| Final Train Loss | 0.8696     |
| Final Val Loss   | 0          |


## Human Evaluation CSV

In [25]:
import pandas as pd

print("\n=== Human Evaluation CSV (First 5 Rows) ===")

csv_path = "submission_artifacts/bengali_empathy_human_eval.csv"
df_human_eval = pd.read_csv(csv_path)

display(df_human_eval.head())


=== Human Evaluation CSV (First 5 Rows) ===


,input_text,response_text,Empathetic_Score (1-5),Reviewer_Notes
0,আমি এই বছর রসায়ন ডিগ্রি নিয়ে স্নাতক হয়েছি।,কোন আশ্চর্য বিষয়! আপনি এখন একজন রসায়ন বিজ্ঞানী!,NaN,NaN
1,"প্যানকেক, ডিম, টোস্ট...আমার প্রিয়",আমারও! আমি সবসময় প্যানকেক এবং টোস্ট চাই,NaN,NaN
2,তাই অন্য সপ্তাহে আমরা সাঁতার কাটছিলাম এবং আমার...,কখন সে আবার তার সাথে পুলে পড়বে?,NaN,NaN
3,অবশ্যই! আমি সম্ভবত কিছু Netflixও দেখব!,এটা কখন?,NaN,NaN
4,আমরা বড় ছাত্র ঋণ প্রদান করে উদযাপন করতে হবে হ...,আমি এটাই বলতে চাই যে আমরা পরিশোধ করেছি তাই আমর...,NaN,NaN


In [ ]:
# !rm -rf /kaggle/working/lora_model
# !rm -rf ~/.cache/huggingface/hub/*